# E18 Cortex scATAC → h5ad

Convert the 10x ATAC peak matrix into an AnnData object and merge QC metrics.


In [1]:
import os
import h5py
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csc_matrix


In [2]:
ROOT = '/home/nakagawa/datasets/downloads_but_processed/E18_cortex_scATAC'

PEAK_H5 = os.path.join(ROOT, 'filtered_peak_bc_matrix.h5')
QC_CSV = os.path.join(ROOT, 'singlecell.csv.gz')
OUT_H5AD = os.path.join(ROOT, 'E18_cortex_scATAC.h5ad')

print('PEAK_H5 =', PEAK_H5)
print('QC_CSV  =', QC_CSV)
print('OUT_H5AD=', OUT_H5AD)


PEAK_H5 = /home/nakagawa/datasets/downloads_but_processed/E18_cortex_scATAC/filtered_peak_bc_matrix.h5
QC_CSV  = /home/nakagawa/datasets/downloads_but_processed/E18_cortex_scATAC/singlecell.csv.gz
OUT_H5AD= /home/nakagawa/datasets/downloads_but_processed/E18_cortex_scATAC/E18_cortex_scATAC.h5ad


In [3]:
def decode(x):
    return x.decode() if isinstance(x, bytes) else str(x)

def load_peak_matrix(h5_file):

    with h5py.File(h5_file, 'r') as f:

        g = f['matrix']

        X = csc_matrix(
            (
                g['data'][:],
                g['indices'][:],
                g['indptr'][:]
            ),
            shape=tuple(g['shape'][:])
        )

        barcodes = [decode(x) for x in g['barcodes'][:]]
        peaks = [decode(x) for x in g['features']['name'][:]]

    adata = ad.AnnData(X=X.T)
    adata.obs_names = barcodes
    adata.var_names = peaks

    return adata


In [4]:
# Test load
adata = load_peak_matrix(PEAK_H5)
print(adata)
adata


AnnData object with n_obs × n_vars = 5533 × 168175


AnnData object with n_obs × n_vars = 5533 × 168175

In [5]:
# Parse peak coordinates
chrom = []
start = []
end = []

for peak in adata.var_names:
    c, pos = peak.split(':')
    s, e = pos.split('-')

    chrom.append(c)
    start.append(int(s))
    end.append(int(e))

adata.var['chrom'] = chrom
adata.var['start'] = start
adata.var['end'] = end

adata.var.head()


,chrom,start,end
chr1:3094517-3095438,chr1,3094517,3095438
chr1:3113497-3114022,chr1,3113497,3114022
chr1:3119287-3121810,chr1,3119287,3121810
chr1:3181055-3181563,chr1,3181055,3181563
chr1:3198052-3198900,chr1,3198052,3198900


In [6]:
# Load QC table
qc = pd.read_csv(QC_CSV, compression='gzip')

print(qc.shape)
qc.head()


(535472, 18)


,barcode,total,duplicate,chimeric,unmapped,lowmapq,mitochondrial,passed_filters,cell_id,is__cell_barcode,TSS_fragments,DNase_sensitive_region_fragments,enhancer_region_fragments,promoter_region_fragments,on_target_fragments,blacklist_region_fragments,peak_region_fragments,peak_region_cutsites
0,NO_BARCODE,5075867,1654026,31277,1338377,223716,30480,1797991,NaN,0,0,0,0,0,0,0,0,0
1,AAACGAAAGAAACGCC-1,13,1,0,7,0,0,5,NaN,0,2,3,1,2,3,0,3,6
2,AAACGAAAGAAAGCAG-1,3,0,0,3,0,0,0,NaN,0,0,0,0,0,0,0,0,0
3,AAACGAAAGAAATACC-1,8,1,0,7,0,0,0,NaN,0,0,0,0,0,0,0,0,0
4,AAACGAAAGAAATCTG-1,2,1,0,0,0,0,1,NaN,0,1,0,0,1,1,0,1,2


In [7]:
# Keep only cell barcodes present in matrix
qc = qc.set_index('barcode')

qc = qc.loc[
    qc.index.intersection(adata.obs_names)
]

print(qc.shape)


(5533, 17)


In [8]:
# Attach QC metrics
adata.obs = adata.obs.join(qc, how='left')
adata


AnnData object with n_obs × n_vars = 5533 × 168175
    obs: 'total', 'duplicate', 'chimeric', 'unmapped', 'lowmapq', 'mitochondrial', 'passed_filters', 'cell_id', 'is__cell_barcode', 'TSS_fragments', 'DNase_sensitive_region_fragments', 'enhancer_region_fragments', 'promoter_region_fragments', 'on_target_fragments', 'blacklist_region_fragments', 'peak_region_fragments', 'peak_region_cutsites'
    var: 'chrom', 'start', 'end'

In [9]:
# Store fragment file path
adata.uns['fragments_file'] = os.path.join(
    ROOT,
    'fragments.tsv.gz'
)

adata.uns['dataset'] = 'E18_cortex_scATAC'


In [10]:
# Save
adata.write_h5ad(
    OUT_H5AD,
    compression='gzip'
)

print('WROTE:', OUT_H5AD)


WROTE: /home/nakagawa/datasets/downloads_but_processed/E18_cortex_scATAC/E18_cortex_scATAC.h5ad


In [11]:
# Validation
test = ad.read_h5ad(OUT_H5AD, backed='r')

print(test)
print('obs =', test.obs.shape)
print('var =', test.var.shape)


AnnData object with n_obs × n_vars = 5533 × 168175 backed at '/home/nakagawa/datasets/downloads_but_processed/E18_cortex_scATAC/E18_cortex_scATAC.h5ad'
    obs: 'total', 'duplicate', 'chimeric', 'unmapped', 'lowmapq', 'mitochondrial', 'passed_filters', 'cell_id', 'is__cell_barcode', 'TSS_fragments', 'DNase_sensitive_region_fragments', 'enhancer_region_fragments', 'promoter_region_fragments', 'on_target_fragments', 'blacklist_region_fragments', 'peak_region_fragments', 'peak_region_cutsites'
    var: 'chrom', 'start', 'end'
    uns: 'dataset', 'fragments_file'
obs = (5533, 17)
var = (168175, 3)
